# Stage 2 Regularization Search Overview

Overview notebook for Stage 2 regularization search on `target_log_return_20m` with `price_hmm_n4` features.

This notebook is partial-run aware: if `stage2_results.parquet` is not present yet, it loads all available `jobs/*/metrics.json` rows.

## Current S3 Status

At notebook creation time, S3 contained a Stage 2 `main` run at `20260711_100640`: 176 completed jobs out of 288 planned jobs, with no final `stage2_results.parquet` yet. Re-run the notebook after completion; it will automatically switch to the final parquet table when available.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "build_price_feature_day.py").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client
from analysis.stage1_model_search.stage1_analysis_tools import (
    add_composite_score,
    early_stop_curve,
    hyperparameter_importance,
    load_search_results,
    parameter_columns,
    parameter_correlations,
    parameter_summary,
    pareto_frontier,
    plot_early_stop,
    plot_heatmap,
    plot_metric_distributions,
    plot_pareto,
    plot_param_distribution,
    plot_violin_distribution,
    read_parquet,
    run_overview,
    stability_table,
    top_tables,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 240)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
BUCKET = "binance-data-downloader"
DATASET_PREFIX = "dataset_target_20/with_price_hmm_n4"
RESULTS_SUBDIR = "stage2_regularization_search"
RUN_ID = "latest"  # or "20260711_100640"

s3 = make_s3_client()
loaded = load_search_results(
    s3=s3,
    bucket=BUCKET,
    dataset_prefix=DATASET_PREFIX,
    results_subdir=RESULTS_SUBDIR,
    run_id=RUN_ID,
    results_filename="stage2_results.parquet",
)

results = add_composite_score(loaded.results)
run_config = loaded.run_config

print(loaded.active_s3_uri)
print(f"loaded_from_final_table={loaded.loaded_from_final_table}")
print(f"rows={len(results):,}, expected={run_config.get('jobs')}")
run_config

## 1. Completion And Cost

In [ ]:
if results.empty:
    raise RuntimeError("No Stage 2 rows were loaded. Check RUN_ID and S3 prefix.")

overview = run_overview(results, run_config)
overview

In [ ]:
family_loss_summary = (
    results.groupby(["model_family", "loss_function"], dropna=False)
    .agg(
        jobs=("job_id", "count"),
        best_RMSE=("RMSE", "min"),
        median_RMSE=("RMSE", "median"),
        best_MAE=("MAE", "min"),
        best_DA_025=("Direction_Accuracy_0.25%", "max"),
        median_train_time=("train_time", "median"),
        total_train_time_hours=("train_time", lambda value: value.sum() / 3600),
        median_model_size_mb=("model_size_mb", "median"),
    )
    .reset_index()
    .sort_values("best_RMSE")
)
family_loss_summary

In [ ]:
plot_metric_distributions(results)

## 2. Top Models

In [ ]:
tops = top_tables(results, n=15)
for name, table in tops.items():
    print(f"\n{name}")
    display(table)

In [ ]:
leader_hits = []
for board_name, table in tops.items():
    for job_id in table["job_id"].tolist():
        leader_hits.append({"leaderboard": board_name, "job_id": job_id})
leader_hits = pd.DataFrame(leader_hits)
leader_consensus = (
    leader_hits.groupby("job_id")
    .agg(hits=("leaderboard", "count"), leaderboards=("leaderboard", lambda value: ", ".join(sorted(value))))
    .reset_index()
    .merge(results, on="job_id", how="left")
    .sort_values(["hits", "RMSE"], ascending=[False, True])
)
leader_consensus[["job_id", "hits", "leaderboards", "RMSE", "MAE", "Direction_Accuracy_0.25%", "train_time", "params"]].head(30)

## 3. Regularization Parameters

Stage 2 varies regularization around Stage 1 architecture seeds. The most important question is whether regularization consistently improves the stable Stage 1 region or whether gains are just isolated noise.

In [ ]:
param_cols = parameter_columns(results)
stage2_params = [
    "param_depth",
    "param_learning_rate",
    "param_l2_leaf_reg",
    "param_bagging_temperature",
    "param_rsm",
    "param_random_strength",
]
stage2_params = [column for column in stage2_params if column in results.columns]
stage2_params

In [ ]:
for param in stage2_params:
    print(f"\n{param}")
    display(parameter_summary(results, param, metric="RMSE", top_n=15))
    plot_param_distribution(results, param, metrics=("RMSE", "MAE", "Direction_Accuracy_0.25%"))
    plot_violin_distribution(results, param, metric="RMSE")

## 4. Pairwise Effects

In [ ]:
pairs = [
    ("param_bagging_temperature", "param_rsm"),
    ("param_bagging_temperature", "param_random_strength"),
    ("param_rsm", "param_random_strength"),
    ("param_depth", "param_bagging_temperature"),
    ("param_learning_rate", "param_bagging_temperature"),
    ("param_l2_leaf_reg", "param_rsm"),
]

for row_param, col_param in pairs:
    if row_param in results.columns and col_param in results.columns:
        pivot = results.pivot_table(index=row_param, columns=col_param, values="RMSE", aggfunc="median")
        display(pivot)
        plot_heatmap(pivot, title=f"Median RMSE: {row_param} x {col_param}")

## 5. Seed Architecture Comparison

In [ ]:
seed_columns = ["param_depth", "param_learning_rate", "param_l2_leaf_reg"]
seed_columns = [column for column in seed_columns if column in results.columns]
seed_summary = (
    results.groupby(seed_columns, dropna=False)
    .agg(
        jobs=("job_id", "count"),
        best_RMSE=("RMSE", "min"),
        median_RMSE=("RMSE", "median"),
        best_DA_025=("Direction_Accuracy_0.25%", "max"),
        median_train_time=("train_time", "median"),
    )
    .reset_index()
    .sort_values(["best_RMSE", "median_RMSE"])
)
seed_summary

## 6. Pareto And Stability

In [ ]:
frontier = pareto_frontier(results, quality_metric="RMSE", cost_metric="train_time")
pareto_columns = ["job_id", "RMSE", "MAE", "Direction_Accuracy_0.25%", "train_time", "model_size_mb", "params"]
display(frontier.loc[:, pareto_columns])
plot_pareto(results, frontier, quality_metric="RMSE", cost_metric="train_time")

In [ ]:
stability_table(results, metric="RMSE", sizes=(5, 10, 20, 30, 50))

## 7. Hyperparameter Importance And Correlations

In [ ]:
importance, surrogate_model, surrogate_matrix = hyperparameter_importance(
    results,
    metric="RMSE",
    param_cols=stage2_params,
)
importance

In [ ]:
parameter_correlations(results)

## 8. HMM Feature Importance

In [ ]:
def load_feature_importance(row: pd.Series) -> pd.DataFrame:
    key = row.get("feature_importance_key")
    if not isinstance(key, str) or not key:
        return pd.DataFrame()
    frame = read_parquet(s3, BUCKET, key)
    if frame.empty:
        return frame
    frame = frame.copy()
    frame["job_id"] = row["job_id"]
    frame["RMSE"] = row["RMSE"]
    return frame


top_for_importance = results.sort_values(["RMSE", "MAE", "job_id"]).head(15)
feature_importance = pd.concat(
    [load_feature_importance(row) for _, row in top_for_importance.iterrows()],
    ignore_index=True,
) if len(top_for_importance) else pd.DataFrame()

if feature_importance.empty:
    print("No feature importance data loaded yet.")
else:
    feature_importance["importance_abs"] = feature_importance["importance"].abs()
    aggregate_importance = (
        feature_importance.groupby("feature", as_index=False)
        .agg(mean_abs_importance=("importance_abs", "mean"), jobs=("job_id", "nunique"))
        .sort_values("mean_abs_importance", ascending=False)
    )
    display(aggregate_importance.head(30))
    display(aggregate_importance.loc[aggregate_importance["feature"].str.startswith("price_hmm_n4_")])

## 9. Early-Stop Diagnostic

In [ ]:
curve = early_stop_curve(results, metric="RMSE")
display(curve.head(20))
display(curve.tail(20))
plot_early_stop(curve, metric="RMSE")

## 10. Stage 3 Candidate Selection

In [ ]:
selection_columns = [
    "job_id",
    "RMSE",
    "MAE",
    "Direction_Accuracy_0.25%",
    "CompositeScore",
    "train_time",
    "model_size_mb",
    "params",
]

stage3_by_rmse = results.sort_values(["RMSE", "MAE", "job_id"]).loc[:, selection_columns].head(10)
stage3_by_direction = results.sort_values(
    ["Direction_Accuracy_0.25%", "RMSE", "job_id"],
    ascending=[False, True, True],
).loc[:, selection_columns].head(10)
stage3_by_composite = results.sort_values(
    ["CompositeScore", "RMSE", "job_id"],
    ascending=[False, True, True],
).loc[:, selection_columns].head(10)

print("Top by RMSE")
display(stage3_by_rmse)
print("Top by Direction Accuracy 0.25%")
display(stage3_by_direction)
print("Top by Composite Score")
display(stage3_by_composite)

Recommended Stage 3 policy:

- Use the Top RMSE and Top Composite tables as the main candidate source.
- Keep separate direction-accuracy candidates if trading direction matters more than point RMSE.
- If the run is still partial, do not finalize Stage 3 until `completed_jobs == run_config['jobs']` or the remaining seed regions are known to be irrelevant.